# 🗂️ Notebook 2: Collaborative Whiteboard — Data Model & APIs


## 🛠️ Setup

```bash
cd 06-system-designs/collaborative-whiteboard
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🧩 Entities

Every collaborative app boils down to a few core entities. Keep them **small**, **typed**, and **versioned**.

| Entity | Purpose | Durable? |
|---|---|---|
| **Board** | The shared canvas | ✅ yes |
| **Shape** | One object on the canvas (rect, circle, line, text…) | ✅ yes (as part of board state) |
| **Op** | A single edit (`add` / `update` / `delete` a shape) | ✅ append-only log |
| **Presence** | Live cursor / selection for one user | ❌ ephemeral (pub/sub only) |
| **Snapshot** | Full board state at Lamport=N | ✅ S3-style blob |

> 💡 **Why distinguish ops from snapshots?** Ops are *how edits flow*. Snapshots are *how new joiners catch up fast*. Without snapshots, a late joiner on a board with 1M ops would have to replay all of them.


## 🧪 Pydantic models

We use **pydantic** so the model is self-validating. A malformed op from a client is rejected at the edge instead of corrupting state downstream.


In [1]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError

# --- Shape: a single thing on the canvas ---
class Shape(BaseModel):
    id: str
    kind: Literal["rect", "circle", "line", "text"]
    x: float
    y: float
    w: float = 0
    h: float = 0
    color: str = "#000000"
    text: Optional[str] = None  # only for kind == "text"

# --- Op: one edit. This is what flies over the WebSocket. ---
class Op(BaseModel):
    board_id: str
    actor: str                  # user id
    lamport: int = Field(ge=0)  # logical clock (see notebook 3)
    kind: Literal["add", "update", "delete"]
    shape: Optional[Shape] = None
    shape_id: Optional[str] = None  # used when kind == "delete"

# --- Presence: ephemeral cursor position ---
class Presence(BaseModel):
    board_id: str
    actor: str
    x: float
    y: float
    selection: list[str] = []   # shape ids currently selected

s = Shape(id="s1", kind="rect", x=10, y=20, w=50, h=30, color="#ff8800")
op = Op(board_id="b1", actor="alice", lamport=1, kind="add", shape=s)
print(op.model_dump_json(indent=2))


{
  "board_id": "b1",
  "actor": "alice",
  "lamport": 1,
  "kind": "add",
  "shape": {
    "id": "s1",
    "kind": "rect",
    "x": 10.0,
    "y": 20.0,
    "w": 50.0,
    "h": 30.0,
    "color": "#ff8800",
    "text": null
  },
  "shape_id": null
}


### ✅ Let's see validation catch a bad op

Clients are untrusted. Pydantic makes it easy to reject garbage at the gateway.


In [2]:
# 1) Wrong shape kind
try:
    Op(board_id="b1", actor="alice", lamport=1, kind="add",
       shape=Shape(id="s2", kind="triangle", x=0, y=0))  # not a valid kind
except ValidationError as e:
    print("rejected (bad kind):", e.errors()[0]["msg"])

# 2) Negative lamport clock
try:
    Op(board_id="b1", actor="alice", lamport=-1, kind="add",
       shape=Shape(id="s3", kind="rect", x=0, y=0))
except ValidationError as e:
    print("rejected (negative lamport):", e.errors()[0]["msg"])

# 3) delete without shape_id — add a custom rule.
from pydantic import model_validator

class StrictOp(Op):
    @model_validator(mode="after")
    def _check(self):
        if self.kind == "delete" and not self.shape_id:
            raise ValueError("delete op requires shape_id")
        if self.kind in ("add", "update") and self.shape is None:
            raise ValueError(f"{self.kind} op requires shape")
        return self

try:
    StrictOp(board_id="b1", actor="alice", lamport=1, kind="delete")
except ValidationError as e:
    print("rejected (delete w/o shape_id):", e.errors()[0]["msg"])


rejected (bad kind): Input should be 'rect', 'circle', 'line' or 'text'
rejected (negative lamport): Input should be greater than or equal to 0
rejected (delete w/o shape_id): Value error, delete op requires shape_id


## 🌐 APIs

We use **HTTP for one-shot requests** (create board, load snapshot) and **WebSocket for the live stream** (ops + presence). That's the canonical split for real-time apps.

### HTTP

| Method | Path | Purpose |
|---|---|---|
| `POST` | `/boards` | Create a new board |
| `GET`  | `/boards/{id}` | Fetch latest snapshot + `since` cursor (Lamport) |
| `GET`  | `/boards/{id}/ops?since={lamport}` | Fetch ops after a cursor (used on reconnect) |
| `POST` | `/boards/{id}/snapshot` | (internal) trigger a snapshot |

### WebSocket

```
client → server
  { "type": "op", "op": { ...Op... } }
  { "type": "presence", "presence": { ...Presence... } }
  { "type": "hello", "since": 12345 }      // sent on (re)connect

server → client
  { "type": "op", "op": { ... } }          // broadcast
  { "type": "presence", "presence": { ... } }
  { "type": "catchup", "ops": [ ... ] }    // replies to "hello"
  { "type": "ack", "lamport": 12346 }      // server assigned this lamport
```

### 🏚️ Bad vs 🏛️ Good API shapes

| 🏚️ Bad | 🏛️ Good | Why |
|---|---|---|
| `PUT /boards/{id}` with full JSON of the whole board every edit | `POST op` on WebSocket | Full-board every drag is O(board size) per edit — doesn't scale. |
| Server replies with *nothing* after an op | Server returns `ack` with assigned `lamport` | Client needs the Lamport to dedupe echoes and resume on reconnect. |
| Persist presence in DB | Pub/Sub only, TTL after disconnect | Presence is high-churn and worthless once the user leaves. |
| One monolithic `/edit` endpoint | Typed `Op` with `kind` | Makes validation, auditing, and undo trivial. |


## 🕒 Lamport clock — the simplest ordering primitive

Before we jump to CRDTs in notebook 3, get comfortable with **Lamport timestamps**. They give every op a logical time that respects cause-and-effect across machines *without* needing synchronized wall clocks.

Rules:
1. **On local event**: `t = t + 1`.
2. **On receiving a message with stamp `u`**: `t = max(t, u) + 1`.

Combined with a stable tie-breaker (the actor id), Lamport timestamps give a **total order** on ops — exactly what a Last-Writer-Wins CRDT needs.


In [3]:
class Lamport:
    def __init__(self):
        self.t = 0
    def tick(self):                 # local event
        self.t += 1
        return self.t
    def observe(self, other: int):  # received message with timestamp `other`
        self.t = max(self.t, other) + 1
        return self.t

alice, bob = Lamport(), Lamport()
a1 = alice.tick();                 print(f"alice ticks -> {a1}")
b_after_a = bob.observe(a1);       print(f"bob sees alice({a1}) -> {b_after_a}")
b2 = bob.tick();                   print(f"bob ticks -> {b2}")
a_after_b = alice.observe(b2);     print(f"alice sees bob({b2}) -> {a_after_b}")

assert a_after_b > b2 > b_after_a > a1
print("causal order preserved")


alice ticks -> 1
bob sees alice(1) -> 2
bob ticks -> 3
alice sees bob(3) -> 4
causal order preserved


## 📝 Takeaways

- Small, typed models (`Shape`, `Op`, `Presence`) make the contract between client and server crisp.
- HTTP for snapshots + WebSocket for streams is the standard split.
- Always return an **ack with Lamport** — the client needs it to reconcile on reconnect.
- Presence is ephemeral — never persist it.
- Lamport clocks give a cheap, correct total order across actors.

In **notebook 3** we combine Lamport ordering with a map to build a Last-Writer-Wins CRDT, and demonstrate offline merge.
